# Training Log Reader
Parst `traininglog_test.docx` in einen pandas DataFrame und berechnet die Trainingsbelastung.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.traininglog_parser import parse_training_log
from src.load_score import compute_endurance_performance, compute_combined_load

In [ ]:
FILE_PATH = r"e:\GoogleDrive\privat\Code_Repo\AI_Coach\data\traininglog_test.docx"

In [ ]:
# Inspect raw docx structure (diagnostic)
from docx import Document

doc = Document(FILE_PATH)

print("=== PARAGRAPHS ===")
for i, para in enumerate(doc.paragraphs):
    if para.text.strip():
        print(f"[{i}] Style='{para.style.name}' | {para.text}")

print("\n=== TABLES ===")
for t_idx, table in enumerate(doc.tables):
    print(f"\n-- Table {t_idx} ({len(table.rows)} rows x {len(table.columns)} cols) --")
    for row in table.rows:
        print([cell.text.strip() for cell in row.cells])

In [ ]:
df = parse_training_log(FILE_PATH)
display(df)

## Trainingsbelastung

### Kardiovaskuläre Belastung: TRIMP (Banister)

$$\text{TRIMP} = t \cdot \Delta\text{HR} \cdot e^{b \cdot \Delta\text{HR}}, \quad \Delta\text{HR} = \frac{\overline{\text{HR}} - \text{HR}_\text{rest}}{\text{HR}_\text{max} - \text{HR}_\text{rest}}$$

- **$t$**: aktive Trainingszeit [min] — $b$: Geschlechtsfaktor (1,92 = männlich / 1,67 = weiblich)
- Ohne HR-Daten: TRIMP = Dauer [min] als Volumen-Proxy

### Gesamtbelastung (sport-korrigierter TRIMP)

$$\text{Gesamtbelastung} = \text{TRIMP} \times k_\text{Sport}$$

| Sport | $k_\text{Sport}$ | Grund |
|---|---|---|
| Radfahren (Referenz) | 1,0 | – |
| Laufen | 1,3 | Bodenreaktionskräfte, exzentrische Muskelbelastung |
| Schwimmen | 1,1 | Schultergürtel-Overhead |

> **Nächster Schritt:** `compute_combined_load(df_session, df_wellness)` in `src/load_score.py`
> kombiniert TRIMP mit Garmin-Erholungsmetriken (Resting HR, HRV) zu einem Readiness-Score.

In [ ]:
df_session, df_weekly = compute_endurance_performance(df)

print("=== Belastung pro Trainingseinheit ===")
display(df_session)

print("\n=== Wochensummen ===")
display(df_weekly)